In [1]:
# from ultralytics import YOLO
# import cv2
# import winsound
# import datetime


# model = YOLO("yolov8n.pt")

# cap = cv2.VideoCapture("cctv1.mp4")

# import time

# start_time = None
# loitering_threshold = 5  # seconds
# alert_triggered = False

# while True:
    
#     ret, frame = cap.read()

#     # ✅ Stop when video ends
#     if not ret:
#         break

#     person_detected = False

#     # ✅ Resize for better performance
#     frame = cv2.resize(frame, (640, 480))

#     # ✅ Add confidence threshold
#     results = model(frame, conf=0.3)

#     for r in results:
#         for box in r.boxes:
#             cls = int(box.cls[0])

#             # ✅ Detect only person
#             if cls == 0:
#                 person_detected = True

#                 x1, y1, x2, y2 = map(int, box.xyxy[0])

#                 # Draw box
#                 cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

#                 # Label
#                 cv2.putText(frame, "Person", (x1, y1 - 10),
#                 cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

#                 # Start timer
#                 if start_time is None:
#                     start_time = time.time()

#                 elapsed_time = time.time() - start_time

#                 # Show time
#                 cv2.putText(frame, f"Time: {int(elapsed_time)}s",
#                 (x1, y1 - 30),
#                 cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

#                 # Alert
#                 if elapsed_time > loitering_threshold:
#                     cv2.putText(frame, "ALERT: Loitering!",
#                     (50, 50),
#                     cv2.FONT_HERSHEY_SIMPLEX, 1,
#                     (0, 0, 255), 3)
#                     if not alert_triggered:
#                         winsound.Beep(1000, 500)
#                         timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
#                         filename = f"alert_{timestamp}.jpg"
#                         cv2.imwrite(filename, frame)

#                         print(f"Screenshot saved: {filename}")
#                         alert_triggered = True
#     if not person_detected:
#         start_time = None
#         alert_triggered = False

#     cv2.imshow("Detection", frame)

#     # ✅ Control video speed (IMPORTANT)
#     if cv2.waitKey(25) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()

In [1]:
import os
from dotenv import load_dotenv
import google.generativeai as genai
load_dotenv()

key = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=key)
genai_model = genai.GenerativeModel('gemini-2.5-flash')


from ultralytics import YOLO
import cv2
import winsound
import datetime

import mediapipe as mp

mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_draw = mp.solutions.drawing_utils

prev_hand_y = None
aggressive_threshold = 20


yolo_model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture("fight_vid.mp4")

import time

start_time = None
loitering_threshold = 5  # seconds
alert_triggered = False

aggressive_alert_triggered = False

while True:
    
    ret, frame = cap.read()

    # ✅ Stop when video ends
    if not ret:
        break

    person_detected = False

    # ✅ Resize for better performance
    frame = cv2.resize(frame, (640, 480))

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results_pose = pose.process(rgb_frame)
    if results_pose.pose_landmarks:
        mp_draw.draw_landmarks(frame, results_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS)
    aggressive_detected = False

    if results_pose.pose_landmarks:
        landmarks = results_pose.pose_landmarks.landmark

        # Get important points
        left_wrist = landmarks[mp_pose.PoseLandmark.LEFT_WRIST]
        right_wrist = landmarks[mp_pose.PoseLandmark.RIGHT_WRIST]
        left_shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER]
        right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER]

        # Convert to pixel values
        lw_y = left_wrist.y * frame.shape[0]
        rw_y = right_wrist.y * frame.shape[0]
        ls_y = left_shoulder.y * frame.shape[0]
        rs_y = right_shoulder.y * frame.shape[0]

        # 🥊 Condition 1: Hands raised (above shoulders)
        if lw_y < ls_y or rw_y < rs_y:
            aggressive_detected = True

        # 🏃 Condition 2: Sudden movement
        if prev_hand_y is not None:
            movement = abs(lw_y - prev_hand_y)
            if movement > aggressive_threshold:
                aggressive_detected = True

        prev_hand_y = lw_y

    # ✅ Add confidence threshold
    results = yolo_model(frame, conf=0.3)

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])

            # ✅ Detect only person
            if cls == 0:
                person_detected = True

                x1, y1, x2, y2 = map(int, box.xyxy[0])

                # Draw box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

                # Label
                cv2.putText(frame, "Person", (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

                # Start timer
                if start_time is None:
                    start_time = time.time()

                elapsed_time = time.time() - start_time

                # Show time
                cv2.putText(frame, f"Time: {int(elapsed_time)}s",
                (x1, y1 - 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

                # Alert
                if elapsed_time > loitering_threshold:
                    cv2.putText(frame, "ALERT: Loitering!",
                    (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (0, 0, 255), 3)
                    if not alert_triggered:
                        winsound.Beep(1000, 500)
                        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                        filename = f"alert_{timestamp}.jpg"
                        cv2.imwrite(filename, frame)

                        print(f"Screenshot saved: {filename}")
                        alert_triggered = True
    if not person_detected:
        start_time = None
        alert_triggered = False
        aggressive_alert_triggered = False 
    if aggressive_detected:
        cv2.putText(frame, "ALERT: Aggressive Behavior!",
                (50, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 1,
                (0, 0, 255), 3)

        if not aggressive_alert_triggered:
            winsound.Beep(1000, 500)

            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"aggressive_{timestamp}.jpg"
            cv2.imwrite(filename, frame)

            print("Aggressive behavior detected!")

            from PIL import Image

            img = Image.open(filename)

            response = genai_model.generate_content(
            ["Describe what suspicious activity is happening in this CCTV image.", img]
            )

            print("AI:", response.text)

            aggressive_alert_triggered = True

    cv2.imshow("Detection", frame)

    # ✅ Control video speed (IMPORTANT)
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

c:\DL Projects\Ai_crime_detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hp\AppData\Local\Temp\ipykernel_9332\375540954.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai



0: 480x640 2 persons, 1 car, 1 truck, 1318.6ms
Speed: 10.0ms preprocess, 1318.6ms inference, 48.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 1 car, 1 truck, 75.6ms
Speed: 2.2ms preprocess, 75.6ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)



c:\DL Projects\Ai_crime_detector\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


0: 480x640 2 persons, 1 car, 1 truck, 61.1ms
Speed: 1.6ms preprocess, 61.1ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 car, 1 truck, 63.6ms
Speed: 2.1ms preprocess, 63.6ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 1 car, 1 truck, 77.8ms
Speed: 1.6ms preprocess, 77.8ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)
Aggressive behavior detected!
AI: The image displays **aggressive behavior** between two individuals.

Here's a breakdown of the suspicious activity:

1.  **Direct Alert:** The most prominent indicator is the red text overlay stating "ALERT: Aggressive Behavior!"
2.  **Physical Engagement:** Two individuals are shown in very close physical proximity, appearing to be grappling, pushing, or struggling with each other. Their arms are raised and seem to be engaged around each other's upper bodies/shoulders.
3.  **Body Posture:** The body language of both individuals sug